***
# Process ASPM airport data

This notebook prepares hourly Aviation System Performance Metrics (ASPM) data for the capstone project. It loads one airport and year, removes empty or unneeded columns, and saves a smaller file for later cleaning and feature engineering.

> ASPM provides airport-level conditions rather than one row per flight. These hourly values can add information about scheduled traffic and recent airport delays when they are joined to the flight data.
***

In [80]:
YEAR = 2024

# AIRPORT = "JFK"
# AIRPORT = "ORD"
AIRPORT = "ATL"

RAW_DATA_FILE = f"../data/aspm/raw/aspm_output/run_{YEAR}_{AIRPORT}/aspm_{YEAR}_{AIRPORT}.csv"
PROCESSED_DATA_FILE = f"../data/aspm/processed/{AIRPORT}_{YEAR}.csv"

***
## Load the raw ASPM file

`YEAR` and `AIRPORT` select the raw input file and the processed output location. Load the CSV into a pandas dataframe so its columns can be reviewed and reduced.
***

In [81]:
import pandas as pd

df = pd.read_csv(RAW_DATA_FILE)

***
## Remove columns with no values

A column that is missing for every hour cannot help the analysis. Print these column names before removing them so differences in another airport or year are easy to see.
***

In [82]:
null_only_columns = df.columns[df.isna().all()].tolist()

print(f"Columns containing only null/NaN values: {len(null_only_columns)}")
print(null_only_columns)

df = df.drop(columns=null_only_columns)
df.shape

Columns containing only null/NaN values: 0
[]


(8769, 18)

***
## Remove fields that are not needed

Remove supporting calculation fields, on-time percentages, and average-delay measures that are not part of the planned feature set. The remaining data keeps hourly scheduled traffic and selected airport delay measures for later use.

`errors="ignore"` allows this step to run when a source file does not contain every listed column. It also hides misspelled column names, so review the list when the source data or project needs change.
***

In [83]:
columns_to_drop = [
    "Departures For Metric Computation",
    "Arrivals For Metric Computation",
    "% On-Time Gate Departures",
    "% On-Time Airport Departures",
    "% On-Time Gate Arrivals",
    "Average Gate Departure Delay",
    "Average Airport Departure Delay",
    "Average Block Delay",
    "Average Gate Arrival Delay"
    ]

In [84]:
df = df.drop(columns=columns_to_drop, errors="ignore")

***
## Save the processed ASPM file

Create the output folder if it does not exist, then save the processed dataframe without the pandas index. This is an intermediate file. Dates, hours, duplicate checks, numeric validation, and final features will be handled later.

> **Modeling warning:** scheduled arrival and departure counts are generally known ahead of time. Average taxi and airborne delay values for the current hour may not be known when a prediction is made, so use earlier-hour values or other lagged features to avoid using future information.
***

In [85]:
from pathlib import Path

Path(PROCESSED_DATA_FILE).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_FILE, index=False)